# QM 640 Capstone — Step 4: Return-Series Collection (Yahoo Finance)

For every screened, confirmed event, pulls daily stock returns for the
event firm and the S&P 500 market index (both cap-weighted and equal-
weighted, for the Synopsis's robustness check), covering the estimation
window through the long event window.

**Run this only after Step 3 (manual screening) and Step 3b (ticker
mapping) are both complete and pushed to the repo.**

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [1]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 978, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 978 (delta 71), reused 110 (delta 41), pack-reused 813 (from 1)
Receiving objects: 100% (978/978), 7.54 MiB | 7.96 MiB/s, done.
Resolving deltas: 100% (512/512), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [2]:
!pip install -q pandas numpy yfinance

## Cell 3 — Configuration

In [3]:
import os

SCREENED_FILE = os.path.join(BASE_DIR, "data/raw/screening_worksheet.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "data/processed/returns")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MARKET_TICKER = "^GSPC"        # S&P 500 (cap-weighted, primary benchmark)
MARKET_TICKER_EW = "^SP500EW"  # S&P 500 Equal Weight (robustness check)

# Calendar-day buffer (Yahoo Finance indexes by calendar date, not trading day)
CALENDAR_DAYS_BEFORE = 230   # covers 150 trading days back + buffer
CALENDAR_DAYS_AFTER = 45     # covers the 30-day long event window + buffer

## Cell 4 — Load confirmed events (post-screening)

In [4]:
import pandas as pd


def load_confirmed_events():
    df = pd.read_csv(SCREENED_FILE)
    df = df[df["is_genuine_ai_event"].astype(str).str.upper() == "Y"]
    df = df[df["confounding_event_flag"].astype(str).str.upper() != "Y"]
    df = df[df["trading_halt_flag"].astype(str).str.upper() != "Y"]
    df = df[df["sufficient_history_flag"].astype(str).str.upper() == "Y"]
    df = df.dropna(subset=["ticker"])  # ticker comes pre-merged from Step 3b
    df["file_date"] = pd.to_datetime(df["file_date"])
    df["event_id"] = df["accession_no"]
    print(f"Confirmed events after screening: {len(df)}")
    return df


events = load_confirmed_events()
events.head()

Confirmed events after screening: 450


,index,accession_no,query,cik,company_name,form_type,file_date,adsh,file_name,is_genuine_ai_event,...,confounding_event_flag,trading_halt_flag,sufficient_history_flag,exclude_reason,filing_url,screener_notes,item_codes,item_in_scope,ticker,event_id
1522,1522,0001493152-24-004894:ex99-1.htm,"""AI-based""",278165,OMNIQ Corp. (OMQS) (CIK 0000278165),8-K,2024-02-05,0001493152-24-004894,NaN,Y,...,N,N,Y,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,"1.01,2.03,7.01,9.01",Y,OMQS,0001493152-24-004894:ex99-1.htm
1539,1539,0001493152-24-005096:ex99-2.htm,"""AI-driven""",1829247,"BullFrog AI Holdings, Inc. (BFRG, BFRGW) (CI...",8-K,2024-02-06,0001493152-24-005096,NaN,Y,...,N,N,Y,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,"1.01,7.01,9.01",Y,BFRG,0001493152-24-005096:ex99-2.htm
1560,1560,0001907982-24-000015:d-wavezapataaixpressrele.htm,"""generative AI""",1907982,"D-Wave Quantum Inc. (QBTS, QBTS-WT) (CIK 000...",8-K,2024-02-08,0001907982-24-000015,NaN,Y,...,N,N,Y,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,8.01,Y,QBTS,0001907982-24-000015:d-wavezapataaixpressrele.htm
1580,1580,0001493152-24-005796:ex99-1.htm,"""AI-driven""",1498148,Artificial Intelligence Technology Solutions I...,8-K,2024-02-12,0001493152-24-005796,NaN,Y,...,N,N,Y,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,"8.01,9.01",Y,AITX,0001493152-24-005796:ex99-1.htm
1620,1620,0001493152-24-006806:ex99-1.htm,"""AI-powered""",1498148,Artificial Intelligence Technology Solutions I...,8-K,2024-02-15,0001493152-24-006806,NaN,Y,...,N,N,Y,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,"8.01,9.01",Y,AITX,0001493152-24-006806:ex99-1.htm


## Cell 5 — Pull returns for each confirmed event

In [5]:
import yfinance as yf
import time


def flatten_columns(data):
    """Recent yfinance returns multi-level columns even for a single ticker
    (e.g. ('Close', 'AAPL')), which breaks .rename() and other single-level
    operations downstream. Flatten to plain column names."""
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.droplevel(1)
    return data


def pull_returns_for_event(ticker, event_date, market_series, market_ew_series):
    start = event_date - pd.Timedelta(days=CALENDAR_DAYS_BEFORE)
    end = event_date + pd.Timedelta(days=CALENDAR_DAYS_AFTER)

    try:
        hist = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        hist = flatten_columns(hist)
    except Exception as e:
        print(f"  {ticker}: download failed ({e})")
        return None

    if hist.empty or len(hist) < 150:
        print(f"  {ticker}: insufficient history ({len(hist)} rows) - excluded per criteria (d)")
        return None

    hist["daily_return_firm"] = hist["Close"].pct_change()
    hist = hist.join(market_series.rename("daily_return_market"), how="inner")
    hist = hist.join(market_ew_series.rename("daily_return_market_ew"), how="left")

    n_before = (hist.index < event_date).sum()
    hist["trading_day_offset"] = range(-n_before, len(hist) - n_before)
    return hist[["daily_return_firm", "daily_return_market", "daily_return_market_ew",
                 "trading_day_offset"]].dropna(subset=["daily_return_firm"])


overall_start = events["file_date"].min() - pd.Timedelta(days=CALENDAR_DAYS_BEFORE)
overall_end = events["file_date"].max() + pd.Timedelta(days=CALENDAR_DAYS_AFTER)

print("Pulling market index series (S&P 500 cap-weighted + equal-weight) ...")
mkt_data = yf.download(MARKET_TICKER, start=overall_start, end=overall_end,
                        progress=False, auto_adjust=True)
mkt_data = flatten_columns(mkt_data)
mkt = mkt_data["Close"].pct_change()

mkt_ew_data = yf.download(MARKET_TICKER_EW, start=overall_start, end=overall_end,
                           progress=False, auto_adjust=True)
mkt_ew_data = flatten_columns(mkt_ew_data)
mkt_ew = mkt_ew_data["Close"].pct_change()

collected, skipped = 0, 0
for _, row in events.iterrows():
    df = pull_returns_for_event(row["ticker"], row["file_date"], mkt, mkt_ew)
    if df is None:
        skipped += 1
        continue

    out_path = os.path.join(OUTPUT_DIR, f"{row['ticker']}_{row['event_id']}.csv")
    df.to_csv(out_path)
    collected += 1
    time.sleep(0.2)  # be polite to Yahoo Finance

print(f"\nDone. Collected: {collected} | Skipped (excluded/failed): {skipped}")

Pulling market index series (S&P 500 cap-weighted + equal-weight) ...


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HLLK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2024-02-15 00:00:00 -> 2024-11-16 00:00:00) (Yahoo error = "Data doesn\'t exist for startDate = 1707973200, endDate = 1731733200")')


  HLLK: insufficient history (0 rows) - excluded per criteria (d)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ALBT']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-05-27 00:00:00 -> 2026-02-26 00:00:00) (Yahoo error = "Data doesn\'t exist for startDate = 1748318400, endDate = 1772082000")')


  ALBT: insufficient history (0 rows) - excluded per criteria (d)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FGRS']: YFTzMissingError('possibly delisted; no timezone found')


  FGRS: insufficient history (0 rows) - excluded per criteria (d)

Done. Collected: 447 | Skipped (excluded/failed): 3


## Commit and push results back to GitHub

In [6]:
!git -C {BASE_DIR} add "data/processed/returns/"
!git -C {BASE_DIR} commit -m "Step 4: Yahoo Finance return series for confirmed events"
!git -C {BASE_DIR} push

[main 7df01ce] Step 4: Yahoo Finance return series for confirmed events
 447 files changed, 83485 insertions(+)
 create mode 100644 data/processed/returns/AAPI_0001477932-26-001202:aapi_ex1020.htm.csv
 create mode 100644 data/processed/returns/AASP_0001472375-26-000160:exhibit99-1.htm.csv
 create mode 100644 data/processed/returns/ABSI_0001628280-25-001612:final-jpm2025presentatio.htm.csv
 create mode 100644 data/processed/returns/ABSI_0001672688-26-000003:finalabscijpm2026present.htm.csv
 create mode 100644 data/processed/returns/ACRV_0001193125-24-219180:d355136dex992.htm.csv
 create mode 100644 data/processed/returns/ADBE_0000796343-24-000057:adbeex991q124.htm.csv
 create mode 100644 data/processed/returns/ADMA_0001140361-24-008680:ef20022054_ex99-1.htm.csv
 create mode 100644 data/processed/returns/ADTI_0001641172-25-020247:ex99-01.htm.csv
 create mode 100644 data/processed/returns/ADVB_0001213900-26-040030:ea028508601ex10-1.htm.csv
 create mode 100644 data/processed/returns/ADVB_0